## Example of a deterministic attack

In this notebook we present an example of an attack to a particular public key generated by the BASS key generation algorithm.

In [ ]:
import sys
import os

notebook_dir = os.getcwd()
src_dir = os.path.join(notebook_dir, 'src')
sys.path.append(src_dir)

from attack import *
from communications import *
from experiments import *

For illustration purposes, we have computed 'short' public keys using the BASS key generation algorithm, which gives 'short' public keys with a low probability. Empirically, one can obtain a 'short' key with probability approximatelly $1/20$. By 'short', we mean 'polynomials that live on $B_{n}$ for a small $n$'. In our example, 'pk_short0.txt' saves two polynomials in at most $24$ variables, so that the code can run relativelly fast.

In [ ]:
# Read the public key and the polynomial corresponding to the message we have to sign
file_path = 'src/pk_short0.txt'
pk, q = read_pk(file_path)

In [ ]:
# Compute P and phi(P) as specified on section 4.2
p, phi_p = precomputation(pk)
print(f"Problem size: {phi_p._idx.size}")

We can find $\psi$ using the fifo algorithm. The execution time of the following cell is around 10 minutes.

In [ ]:
psi = fifo(p, phi_p)

Alternativelly, we can use the algorithm described on section 6 to find such $\psi$. We estimate the running time to be around 11 minutes.

In [ ]:
# Find psi as specified on section 6
psi = find_psi(p, phi_p)

Now we would like to verify that the computed $\psi$ satisfies the sufficient condition. The printing algorithm is not very efficient - it converts a polynomial expressed on the $e_c$ basis to $b_c$ basis. The estimated time to run the following cell is 2 hours.

In [ ]:
# Check that our psi verifies the sufficient condition
psi_p1 = psi.apply(pk[0])
psi_p1 *= -1
psi_p1 += pk[3]
print(f"psi(P1)-phi(P1) = {psi_p1}")
psi_p2 = psi.apply(pk[1])
psi_p2 *= -1
psi_p2 += pk[4]
print(f"psi(P2)-phi(P2) = {psi_p2}")
psi_p3 = psi.apply(pk[2])
psi_p3 *= -1
psi_p3 += pk[5]
print(f"psi(P2)-phi(P2) = {psi_p3}")

Next, we can try to sign a message. Such a message was previously encoded as a polynomial in $B_n$ and saved in the file 'pk_short0.txt', which we previously read. We first expand our domain, as $q$ might be defined over some variables where $\psi$ is not defined. This may take around 55 hours. Then, applying the automorphism takes less than a minute.

In [ ]:
# Sign
signature = psi.apply(q)

We can now check this signature is accepted most of the time. As discused in our paper, such probability is not equal to one as the verification algorithm is probabilistic, but close to 0.99. In the next cell we run the probbilistic verification $10,000$ times to estimate such probability. You can expect the next cell to run in around 1 hour.

In [ ]:
# Make an estimate of the amount of times our signature is accepted
count = 0
num_experiments = 10000

for i in range(num_experiments):
    if validate(pk, q, signature): count += 1

print(f"psi(Q) is a valid signature with probability approximately {count/num_experiments}")